# Riskwright: Exploratory Data Analysis

Home Credit Default Risk. This notebook covers the dataset summary, data quality,
feature categorization, and five business insights with supporting charts.

Every statistic is computed by `src.data.eda`, the same module the API serves from.
Importing rather than reimplementing means the notebook, the UI, and the README cannot
quote different numbers for the same finding.

**Prerequisites:** Postgres running with the three tables loaded (`docker-compose up`).
Paths are relative, and nothing here is Colab-specific.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.data.eda import (
    business_insights,
    dataset_summary,
    distribution,
    group_default_rate,
)

pd.set_option("display.max_columns", 40)

## 1. Dataset summary

In [ ]:
summary = dataset_summary()

pd.Series({
    "rows": f"{summary['rows']:,}",
    "columns": summary["columns"],
    "defaulted": f"{summary['defaulted']:,}",
    "repaid": f"{summary['repaid']:,}",
    "default rate": f"{summary['default_rate']:.2%}",
    "imbalance ratio": f"{summary['imbalance_ratio']:.2f} to 1",
}).to_frame("value")

The imbalance is the single most important property of this target. A model that
predicts "never defaults" is 92% accurate, which is why accuracy is not reported anywhere
in this project. ROC-AUC and PR-AUC are used instead.

## 2. Data quality and missing values

In [ ]:
missing = pd.DataFrame(summary["missing_top"])
print(f"{summary['columns_with_missing']} of {summary['columns']} columns have missing values")

fig = px.bar(missing.head(15), x="missing_pct", y="column", orientation="h",
             labels={"missing_pct": "% missing", "column": ""})
fig.update_layout(height=460, yaxis={"categoryorder": "total ascending"})
fig

The worst offenders are building-level survey fields, 50 to 70% missing. They are kept:
LightGBM handles missing values natively, and dropping them would discard signal on the
applicants who do have them.

The more dangerous problem is not missingness. It is a sentinel value masquerading as data,
covered in insight 1.

## 3. Feature categorization

| Group | Columns |
|---|---|
| Identity and target | `sk_id_curr`, `target` |
| Demographics | `code_gender`, `days_birth`, `cnt_children`, `cnt_fam_members`, `name_family_status`, `name_education_type`, `name_housing_type` |
| Employment | `days_employed`, `occupation_type`, `organization_type`, `name_income_type`, `amt_income_total` |
| Loan terms | `name_contract_type`, `amt_credit`, `amt_annuity`, `amt_goods_price` |
| External credit signals | `ext_source_1`, `ext_source_2`, `ext_source_3` |
| Assets | `flag_own_car`, `own_car_age`, `flag_own_realty` |
| Contactability | `flag_phone`, `flag_email`, `days_last_phone_change`, `days_id_publish` |
| Region | `region_rating_client`, `region_population_relative`, `reg_city_not_work_city` |
| Social circle | `obs_30_cnt_social_circle`, `def_30_cnt_social_circle` |
| Bureau enquiries | `amt_req_credit_bureau_year` |

The remaining ~60 columns are normalised building statistics and document flags. Retained
for the model, but deliberately not exposed to the chatbot: no business user asks about
`nonlivingapartments_medi`, and every extra column is another chance to pick a plausible
but wrong one.

## 4. Five business insights

In [ ]:
payload = business_insights()
for i, insight in enumerate(payload["insights"], 1):
    print(f"{i}. {insight['title']}")
    print(f"   {insight['finding']}")
    print()


### Insight 1: A sentinel value hides in the employment column

In [ ]:
insight = payload["insights"][0]
print(insight["finding"])
print()
print("So what:", insight["so_what"])

evidence = insight["evidence"]
key = next((k for k in ("bands", "groups", "quartiles") if k in evidence), None)
if key:
    frame = pd.DataFrame(evidence[key])
    display(frame)
    fig = px.bar(frame, x=frame.columns[0], y="default_rate",
                 labels={"default_rate": "Default rate"})
    fig.add_hline(y=payload["base_default_rate"], line_dash="dot",
                  annotation_text="portfolio average")
    fig.update_layout(height=340, yaxis_tickformat=".1%")
    fig.show()

### Insight 2: External credit scores separate risk more than anything the applicant reports

In [ ]:
insight = payload["insights"][1]
print(insight["finding"])
print()
print("So what:", insight["so_what"])

evidence = insight["evidence"]
key = next((k for k in ("bands", "groups", "quartiles") if k in evidence), None)
if key:
    frame = pd.DataFrame(evidence[key])
    display(frame)
    fig = px.bar(frame, x=frame.columns[0], y="default_rate",
                 labels={"default_rate": "Default rate"})
    fig.add_hline(y=payload["base_default_rate"], line_dash="dot",
                  annotation_text="portfolio average")
    fig.update_layout(height=340, yaxis_tickformat=".1%")
    fig.show()

### Insight 3: Education level tracks default risk strongly

In [ ]:
insight = payload["insights"][2]
print(insight["finding"])
print()
print("So what:", insight["so_what"])

evidence = insight["evidence"]
key = next((k for k in ("bands", "groups", "quartiles") if k in evidence), None)
if key:
    frame = pd.DataFrame(evidence[key])
    display(frame)
    fig = px.bar(frame, x=frame.columns[0], y="default_rate",
                 labels={"default_rate": "Default rate"})
    fig.add_hline(y=payload["base_default_rate"], line_dash="dot",
                  annotation_text="portfolio average")
    fig.update_layout(height=340, yaxis_tickformat=".1%")
    fig.show()

### Insight 4: Borrowing a lot relative to income does not predict default

In [ ]:
insight = payload["insights"][3]
print(insight["finding"])
print()
print("So what:", insight["so_what"])

evidence = insight["evidence"]
key = next((k for k in ("bands", "groups", "quartiles") if k in evidence), None)
if key:
    frame = pd.DataFrame(evidence[key])
    display(frame)
    fig = px.bar(frame, x=frame.columns[0], y="default_rate",
                 labels={"default_rate": "Default rate"})
    fig.add_hline(y=payload["base_default_rate"], line_dash="dot",
                  annotation_text="portfolio average")
    fig.update_layout(height=340, yaxis_tickformat=".1%")
    fig.show()

### Insight 5: Younger applicants default substantially more often

In [ ]:
insight = payload["insights"][4]
print(insight["finding"])
print()
print("So what:", insight["so_what"])

evidence = insight["evidence"]
key = next((k for k in ("bands", "groups", "quartiles") if k in evidence), None)
if key:
    frame = pd.DataFrame(evidence[key])
    display(frame)
    fig = px.bar(frame, x=frame.columns[0], y="default_rate",
                 labels={"default_rate": "Default rate"})
    fig.add_hline(y=payload["base_default_rate"], line_dash="dot",
                  annotation_text="portfolio average")
    fig.update_layout(height=340, yaxis_tickformat=".1%")
    fig.show()

## 5. Supporting distributions

In [ ]:
for column in ("age_years", "ext_source_3", "credit_income_ratio"):
    data = distribution(column, bins=25)
    frame = pd.DataFrame(data["bins"])

    fig = go.Figure()
    fig.add_bar(x=frame["bin_start"], y=frame["repaid"], name="Repaid")
    fig.add_bar(x=frame["bin_start"], y=frame["defaulted"], name="Defaulted")
    fig.add_scatter(x=frame["bin_start"], y=frame["default_rate"], name="Default rate",
                    yaxis="y2", line=dict(color="black", dash="dot"))
    fig.update_layout(
        title=f"{column} ({data['note']})",
        barmode="stack", height=380,
        yaxis=dict(title="Applicants"),
        yaxis2=dict(title="Default rate", overlaying="y", side="right", tickformat=".0%"),
    )
    fig.show()

## Conclusions carried into the model

1. `days_employed` needs the sentinel replaced before use, and the fact of the sentinel
   kept as a flag.
2. External credit scores dominate. They drive both the model and the derived rules, and
   their absence is the main coverage risk.
3. The target is imbalanced 11.4 to 1, so `scale_pos_weight` is used and PR-AUC is
   reported alongside ROC-AUC.
4. Leverage does not behave as intuition suggests. A leverage cap would be the wrong
   policy lever; loan term carries more signal.
5. Age and education both separate risk strongly and are both legally sensitive, which is
   why derived rules are presented for review rather than applied automatically.